# Chapter 11: Texture Mapping

Source span: *Fundamentals of Computer Graphics*, Chapter 11, printed pages 255-290, physical PDF pages 272-307.

Chapter goal: make texture mapping inspectable as a pair of mappings and a sampling problem. A texture does not live on the surface by magic. A renderer first evaluates a texture coordinate function from the surface to texture space, then filters a sampled texture over the pixel's footprint. The same machinery later drives material maps, normal and bump maps, displacement maps, shadow maps, environment maps, and procedural solid textures.

The source span emphasizes two recurring hazards: texture coordinates can distort or fold, and point-sampling a detailed texture can alias badly. This notebook uses only synthetic textures so the computations are reproducible and copyright-safe. Every visual is generated from arrays, maps, derivatives, or small geometric models in the notebook itself.

## Translation guide

| Source idea | Computational object used here | Main failure mode to inspect |
| --- | --- | --- |
| Texture coordinate function `phi: S -> T` | Python functions that return `(u, v)` from points or interpolated vertices | thinking of texture mapping as `texture -> surface` instead of the lookup direction needed by a shader |
| Texture lookup | nearest and bilinear samplers over synthetic RGB arrays | off-by-half texel mistakes, wrong wrap mode, or sampling outside the intended domain |
| Planar, spherical, cylindrical, and cube coordinates | analytic maps from 3D positions or directions to 2D texture domains | one-to-many projection, pole distortion, or discontinuities at seams |
| Interpolated mesh coordinates | barycentric interpolation of per-vertex UVs | triangles that cross a seam interpolate the long way unless vertices are duplicated |
| Texture transforms and wrapping modes | affine transforms in homogeneous 2D UV space plus repeat, clamp, and constant extension | wasting texture resolution or accidentally making a pattern tile |
| Pixel footprint | the Jacobian from image coordinates to texture coordinates | filtering as a point sample instead of an average over an area |
| Mipmaps | an image pyramid with trilinear level selection `k = log2(D)` | using square averages for elongated footprints; blurred grazing surfaces |
| Bump and normal maps | height derivatives converted to tangent-frame normals | changing the shading normal without changing the silhouette or parallax |
| Displacement maps | a tessellated surface whose vertices are moved by a height texture | confusing a shading trick with changed geometry |
| Shadow maps | a depth texture rendered from the light, sampled with bias and percentage-closer filtering | self-shadowing acne, edge aliasing, or filtering depths instead of visibility tests |
| Environment maps | a texture over directions on the unit sphere | latitude-longitude pole compression and reflection vectors outside unit length |
| Procedural solid textures | functions `c(p)` and gradient noise over 3D points | using white noise when a smooth spatial field is required |

## Visual storyboard

| Order | Artifact | Concept | Representation and library | Learner inspection target | Validation |
| --- | --- | --- | --- | --- | --- |
| 1 | `texture-coordinate-dataflow.png` | lookup direction and spaces | Matplotlib diagram plus NumPy texture samples | the shader maps a surface point to `(u, v)` before reading the texture | repeat-mode periodicity and lookup bounds |
| 2 | `parameterization-distortion-seams.png` | planar, spherical, cylindrical, and cubemap coordinates | analytic UV maps sampled with NumPy and Matplotlib | closed shapes need seams or non-injective maps; spherical maps shrink texel area near poles | face assignment and pole distortion ratios |
| 3 | `interpolated-uv-seam-wrapping.png` | barycentric UVs, seams, wrapping, and texture transforms | Matplotlib atlas and sampler comparison | duplicated seam vertices prevent interpolation across the whole atlas | UV weight sums, seam jump, repeat/clamp behavior |
| 4 | `affine-vs-perspective-texture.png` | perspective-correct interpolation | rasterized synthetic quad with barycentric weights | `u/w`, `v/w`, and `1/w` division fixes screen-linear UV drift | corrected UV agrees with homogeneous formula; affine error is nonzero |
| 5 | `mipmap-pyramid-lod.png` and `filtering-mipmap-anisotropy.png` | reconstruction, footprints, mipmaps, anisotropic filtering | NumPy mip pyramid, derivative footprints, Matplotlib comparison | point sampling glitters; mipmaps average square footprints; anisotropic sampling keeps long-axis detail | mean preservation, high-frequency energy reduction, LOD range |
| 6 | `bump-displacement-normal-field.png` and `bump-displacement-surface.html` | normal maps, bump maps, and displacement | height derivatives plus Plotly 3D surface | bump changes lighting; displacement changes geometry | unit normals and nonzero displacement range |
| 7 | `shadow-map-bias-pcf.png` | shadow depth lookup, bias, and percentage-closer filtering | synthetic 1D shadow map rendered as image strips and curves | compare depth, not interpolated depth; average 0/1 tests for soft edges | bias removes self-shadowing while keeping occluder shadows |
| 8 | `environment-map-reflection-lookup.png` | environment maps and reflection mapping | synthetic longitude-latitude texture sampled by reflection vectors | illumination lookup depends on direction, not surface position | reflected vectors and UVs stay valid |
| 9 | `procedural-solid-texture-turbulence.png` | 3D stripes, solid noise, and turbulence | vectorized gradient noise and turbulence field | texture can be a function of 3D position rather than a stored 2D image | finite range, decreasing octave amplitudes, nonblank turbulence |

Library routing: NumPy carries the maps, sampling, derivatives, and noise; Matplotlib is used for durable 2D diagrams and image diagnostics; Plotly is used for the 3D displacement surface where rotation helps inspect changed geometry. The chapter is image-geometry heavy, so these are a better fit than a mesh-processing or topology stack.

In [ ]:
from pathlib import Path
import math
import sys

CHAPTER = 11
TOPIC = "chapter-11"
TITLE = "Texture Mapping"
PRINTED_PAGES = "255-290"
PDF_PAGES = "272-307"
SOURCE_SPAN = f"printed pages {PRINTED_PAGES}; physical PDF pages {PDF_PAGES}"

BOOK_ROOT = Path.cwd()
for candidate in [BOOK_ROOT, *BOOK_ROOT.parents]:
    if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils").exists():
        BOOK_ROOT = candidate
        break
else:
    raise RuntimeError("Could not find the FCG book root")

if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / TOPIC
for folder in ["figures", "html", "checks", "tables", "data"]:
    (ARTIFACT_ROOT / folder).mkdir(parents=True, exist_ok=True)

import matplotlib.pyplot as plt
from matplotlib import patches
from matplotlib.colors import ListedColormap
import numpy as np
import pandas as pd
import plotly.graph_objects as go

from utils.artifacts import assert_artifacts, display_artifact, save_image, save_json, save_matplotlib, save_plotly_html, save_table_csv
from utils.notebook_checks import assert_nonblank_image

plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white", "font.size": 9})
COLORS = {
    "ink": "#1f2933",
    "gray": "#6b7280",
    "light": "#eef2f7",
    "blue": "#2f6fbb",
    "teal": "#118c8b",
    "gold": "#d99a22",
    "red": "#c84b31",
    "violet": "#6f5bb8",
    "green": "#3f8f4f",
}

image_paths = []
html_paths = []
table_paths = []
check_paths = []
checks = {}


def rel_book(path):
    p = Path(path)
    if not p.is_absolute() and p.parts and p.parts[0] == "artifacts":
        return p.as_posix()
    try:
        return p.resolve().relative_to(BOOK_ROOT).as_posix()
    except ValueError:
        return p.as_posix()


def remember(path, kind="image"):
    path = Path(path)
    if kind == "image":
        image_paths.append(path)
    elif kind == "html":
        html_paths.append(path)
    elif kind == "table":
        table_paths.append(path)
    elif kind == "check":
        check_paths.append(path)
    else:
        raise ValueError(kind)
    return path


def normalize01(x):
    x = np.asarray(x, dtype=float)
    lo = float(np.nanmin(x))
    hi = float(np.nanmax(x))
    if hi <= lo:
        return np.zeros_like(x)
    return (x - lo) / (hi - lo)


def synthetic_uv_texture(size=256):
    y, x = np.mgrid[0:size, 0:size]
    u = x / (size - 1)
    v = y / (size - 1)
    checker = ((np.floor(u * 8) + np.floor(v * 8)) % 2).astype(float)
    rgb = np.dstack([
        0.18 + 0.58 * u,
        0.20 + 0.55 * v,
        0.30 + 0.38 * checker,
    ])
    grid = ((x % 32) < 2) | ((y % 32) < 2)
    rgb[grid] = np.array([0.02, 0.025, 0.03])
    diag = np.abs(u - v) < 0.012
    rgb[diag] = np.array([0.95, 0.25, 0.08])
    ring = np.abs(np.sqrt((u - 0.70) ** 2 + (v - 0.28) ** 2) - 0.115) < 0.012
    rgb[ring] = np.array([0.05, 0.78, 0.86])
    rgb[(u < 0.10) & (v < 0.10)] = np.array([0.98, 0.84, 0.18])
    rgb[(u > 0.90) & (v > 0.90)] = np.array([0.48, 0.18, 0.86])
    return np.clip(rgb, 0.0, 1.0)


def _wrap_uv(u, v, mode="repeat"):
    u, v = np.broadcast_arrays(np.asarray(u, dtype=float), np.asarray(v, dtype=float))
    mask = np.ones(u.shape, dtype=bool)
    if mode == "repeat":
        return u % 1.0, v % 1.0, mask
    if mode == "clamp":
        return np.clip(u, 0.0, 1.0 - 1e-12), np.clip(v, 0.0, 1.0 - 1e-12), mask
    if mode == "constant":
        mask = (u >= 0.0) & (u < 1.0) & (v >= 0.0) & (v < 1.0)
        return np.clip(u, 0.0, 1.0 - 1e-12), np.clip(v, 0.0, 1.0 - 1e-12), mask
    raise ValueError(f"unknown wrap mode {mode!r}")


def sample_nearest(texture, u, v, mode="repeat", background=(1.0, 1.0, 1.0)):
    texture = np.asarray(texture)
    h, w = texture.shape[:2]
    uu, vv, mask = _wrap_uv(u, v, mode)
    i = np.floor(uu * w).astype(int)
    j = np.floor(vv * h).astype(int)
    i = np.clip(i, 0, w - 1)
    j = np.clip(j, 0, h - 1)
    out = texture[j, i].copy()
    if mode == "constant" and out.ndim >= 1:
        out[~mask] = np.asarray(background, dtype=out.dtype)
    return out


def sample_bilinear(texture, u, v, mode="repeat", background=(1.0, 1.0, 1.0)):
    texture = np.asarray(texture, dtype=float)
    h, w = texture.shape[:2]
    uu, vv, mask = _wrap_uv(u, v, mode)
    x = uu * w - 0.5
    y = vv * h - 0.5
    i0 = np.floor(x).astype(int)
    j0 = np.floor(y).astype(int)
    tx = x - i0
    ty = y - j0
    i1 = i0 + 1
    j1 = j0 + 1
    if mode == "repeat":
        i0 %= w; i1 %= w; j0 %= h; j1 %= h
    else:
        i0 = np.clip(i0, 0, w - 1); i1 = np.clip(i1, 0, w - 1)
        j0 = np.clip(j0, 0, h - 1); j1 = np.clip(j1, 0, h - 1)
    c00 = texture[j0, i0]
    c10 = texture[j0, i1]
    c01 = texture[j1, i0]
    c11 = texture[j1, i1]
    wx0 = (1.0 - tx)[..., None]
    wx1 = tx[..., None]
    wy0 = (1.0 - ty)[..., None]
    wy1 = ty[..., None]
    out = c00 * wx0 * wy0 + c10 * wx1 * wy0 + c01 * wx0 * wy1 + c11 * wx1 * wy1
    if mode == "constant":
        out[~mask] = np.asarray(background, dtype=float)
    return np.clip(out, 0.0, 1.0)


def barycentric_points(points, tri):
    points = np.asarray(points, dtype=float)
    a, b, c = np.asarray(tri, dtype=float)
    v0 = b - a
    v1 = c - a
    v2 = points - a
    den = v0[0] * v1[1] - v1[0] * v0[1]
    w1 = (v2[..., 0] * v1[1] - v1[0] * v2[..., 1]) / den
    w2 = (v0[0] * v2[..., 1] - v2[..., 0] * v0[1]) / den
    w0 = 1.0 - w1 - w2
    return np.stack([w0, w1, w2], axis=-1)


def build_mipmap(texture):
    levels = [np.asarray(texture, dtype=float)]
    current = levels[0]
    while current.shape[0] > 1 and current.shape[1] > 1:
        h, w = current.shape[:2]
        current = current[: h - h % 2, : w - w % 2]
        current = current.reshape(current.shape[0] // 2, 2, current.shape[1] // 2, 2, current.shape[2]).mean(axis=(1, 3))
        levels.append(current)
    return levels


def sample_mipmap_trilinear(levels, u, v, footprint_d):
    u, v, footprint_d = np.broadcast_arrays(np.asarray(u, dtype=float), np.asarray(v, dtype=float), np.asarray(footprint_d, dtype=float))
    max_level = len(levels) - 1
    k = np.clip(np.log2(np.maximum(footprint_d, 1.0)), 0.0, max_level)
    k0 = np.floor(k).astype(int)
    k1 = np.minimum(k0 + 1, max_level)
    t = (k - k0)[..., None]
    out0 = np.zeros(u.shape + (3,), dtype=float)
    out1 = np.zeros_like(out0)
    for level_index, level in enumerate(levels):
        m0 = k0 == level_index
        m1 = k1 == level_index
        if np.any(m0):
            out0[m0] = sample_bilinear(level, u[m0], v[m0], mode="repeat")
        if np.any(m1):
            out1[m1] = sample_bilinear(level, u[m1], v[m1], mode="repeat")
    return out0 * (1.0 - t) + out1 * t


def image_gradient_energy(image):
    image = np.asarray(image, dtype=float)
    gx = np.diff(image, axis=1)
    gy = np.diff(image, axis=0)
    return float(np.mean(np.abs(gx)) + np.mean(np.abs(gy)))

BOOK_ROOT.relative_to(BOOK_ROOT), ARTIFACT_ROOT.relative_to(BOOK_ROOT)

## 1. Texture lookup: the map points backward

A texture lookup starts at the surface point being shaded. The renderer already knows where the point projects in the image; it also needs the surface's texture coordinate function to ask where that point lands in texture space. This direction matters. We usually say that a texture is put on a surface, but the shader evaluates `surface point -> (u, v) -> texel value`.

The synthetic texture below is deliberately diagnostic: grid lines reveal distortion, a diagonal reveals orientation, and colored corner marks make wrapping mistakes obvious.

In [ ]:
texture = synthetic_uv_texture(256)
texture_path = remember(save_image(texture, TOPIC, "synthetic-uv-test-texture.png"), "image")

surface_xy = np.array([[0.10, 0.15], [0.30, 0.70], [0.72, 0.35], [0.86, 0.84]])
uv = np.column_stack([0.12 + 0.76 * surface_xy[:, 0] + 0.09 * surface_xy[:, 1], 0.08 + 0.82 * surface_xy[:, 1]])
uv = np.clip(uv, 0, 1)
lookup_colors = sample_bilinear(texture, uv[:, 0], uv[:, 1], mode="clamp")
periodic_error = float(np.max(np.abs(sample_nearest(texture, uv[:, 0] + 2.0, uv[:, 1] - 3.0) - sample_nearest(texture, uv[:, 0], uv[:, 1]))))
checks.update({
    "texture_lookup_uv_min": float(uv.min()),
    "texture_lookup_uv_max": float(uv.max()),
    "repeat_mode_periodic_error": periodic_error,
    "diagnostic_texture_channel_std_min": float(np.std(texture, axis=(0, 1)).min()),
})

fig, axes = plt.subplots(1, 3, figsize=(12.5, 4.0))
axes[0].set_title("image samples")
axes[0].set_xlim(0, 1); axes[0].set_ylim(0, 1); axes[0].set_aspect("equal")
axes[0].set_xlabel("x_s"); axes[0].set_ylabel("y_s")
axes[0].grid(color="#d7dde6")
axes[0].scatter(surface_xy[:, 0], surface_xy[:, 1], s=190, c=lookup_colors, edgecolor="black", zorder=3)
for k, pnt in enumerate(surface_xy):
    axes[0].text(pnt[0], pnt[1] + 0.055, f"p{k}", ha="center")
axes[1].set_title("surface point carries geometry")
axes[1].set_xlim(0, 1); axes[1].set_ylim(0, 1); axes[1].set_aspect("equal")
patch = patches.Polygon([[0.08, 0.20], [0.88, 0.12], [0.76, 0.78], [0.18, 0.88]], closed=True, facecolor="#eef6f6", edgecolor=COLORS["teal"], lw=2)
axes[1].add_patch(patch)
axes[1].scatter(surface_xy[:, 0], surface_xy[:, 1], s=170, c=lookup_colors, edgecolor="black")
axes[1].text(0.52, 0.50, "normal, material,\ntexture coordinate function", ha="center", va="center")
axes[1].axis("off")
axes[2].set_title("texture space T")
axes[2].imshow(texture, origin="lower", extent=(0, 1, 0, 1))
axes[2].scatter(uv[:, 0], uv[:, 1], s=160, c=lookup_colors, edgecolor="white", linewidth=1.4)
for k, q in enumerate(uv):
    axes[2].text(q[0], q[1] + 0.045, f"(u{k},v{k})", ha="center", color="white", fontsize=8)
axes[2].set_xlabel("u"); axes[2].set_ylabel("v")
axes[2].set_xlim(0, 1); axes[2].set_ylim(0, 1)
fig.suptitle("Texture mapping evaluates p -> phi(p) -> texture sample", y=1.03, fontsize=12)
fig.tight_layout()
dataflow_path = remember(save_matplotlib(fig, TOPIC, "texture-coordinate-dataflow.png"), "image")
plt.close(fig)
display_artifact(dataflow_path, width=860)
display_artifact(texture_path, width=360)
checks["texture_lookup_sample_count"] = int(len(surface_xy))
checks

## 2. Texture coordinate functions and parameterization distortion

A good texture coordinate function balances several incompatible goals: avoid accidental many-to-one mapping, keep local scale nearly constant, preserve small shapes, and keep discontinuities under control. Analytic coordinates make those tradeoffs visible.

Planar projection is cheap and useful for nearly flat pieces, but it is not injective on a closed object. Spherical and cylindrical coordinates use angles and naturally create seams. Cubemaps replace two pole singularities with six square charts and edge seams, reducing distortion for direction maps.

In [ ]:
theta = np.linspace(0.001, math.pi - 0.001, 260)
phi = np.linspace(-math.pi, math.pi, 420)
PHI, THETA = np.meshgrid(phi, theta)
latlong_area = np.sin(THETA)
latlong_stretch = 1.0 / np.maximum(latlong_area, 1e-3)

dirs = np.stack([np.sin(THETA) * np.cos(PHI), np.sin(THETA) * np.sin(PHI), np.cos(THETA)], axis=-1)
abs_dirs = np.abs(dirs)
major = np.argmax(abs_dirs, axis=-1)
sign = np.take_along_axis(np.sign(dirs), major[..., None], axis=-1)[..., 0]
face_id = major * 2 + (sign > 0).astype(int)
face_names = ["-x", "+x", "-y", "+y", "-z", "+z"]
face_counts = {face_names[i]: int(np.sum(face_id == i)) for i in range(6)}

front = np.array([0.35, -0.25, 0.82])
back = np.array([0.35, -0.25, -0.82])
planar_collision_error = float(np.linalg.norm(front[:2] - back[:2]))

fig, axes = plt.subplots(2, 2, figsize=(12.0, 8.2))
ax = axes[0, 0]
im = ax.imshow(np.log10(latlong_stretch), origin="lower", extent=(-180, 180, -90, 90), cmap="magma")
ax.axvline(-180, color="white", lw=1.5, ls="--"); ax.axvline(180, color="white", lw=1.5, ls="--")
ax.set_title("latitude-longitude stretch: poles cost resolution")
ax.set_xlabel("longitude degrees"); ax.set_ylabel("latitude degrees")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.03, label="log10 stretch")
ax = axes[0, 1]
face_cmap = ListedColormap(["#2f6fbb", "#9bc1ff", "#118c8b", "#8fe1d8", "#d99a22", "#ffd37a"])
ax.imshow(face_id, origin="lower", extent=(-180, 180, -90, 90), cmap=face_cmap, vmin=0, vmax=5, interpolation="nearest")
ax.set_title("cubemap face: largest absolute direction component")
ax.set_xlabel("longitude degrees"); ax.set_ylabel("latitude degrees")
for i, name in enumerate(face_names):
    ax.text(-170 + i * 68, 74, name, color="black", bbox=dict(facecolor="white", alpha=0.75, edgecolor="none", pad=1.5), fontsize=8)
ax = axes[1, 0]
ax.add_patch(patches.Circle((0, 0), 1.0, fill=False, lw=2, edgecolor=COLORS["ink"]))
ax.plot([front[0], back[0]], [front[2], back[2]], "o", color=COLORS["red"])
ax.annotate("same planar (u,v)", xy=(front[0], front[2]), xytext=(-0.75, 0.45), arrowprops=dict(arrowstyle="->", color=COLORS["red"]), color=COLORS["red"])
ax.annotate("projection direction", xy=(front[0], 0.05), xytext=(front[0], 1.25), arrowprops=dict(arrowstyle="->", color=COLORS["blue"]), ha="center", color=COLORS["blue"])
ax.set_title("planar projection is one-to-many on closed shapes")
ax.set_xlabel("x"); ax.set_ylabel("z")
ax.set_xlim(-1.25, 1.25); ax.set_ylim(-1.25, 1.35); ax.set_aspect("equal")
ax = axes[1, 1]
v = np.linspace(0.001, 0.999, 400)
theta_v = math.pi * (1.0 - v)
area_scale = np.sin(theta_v)
ax.plot(v, area_scale, color=COLORS["teal"], lw=2, label="surface area per equal UV cell")
ax.plot(v, 1.0 / np.maximum(area_scale, 1e-3), color=COLORS["gold"], lw=2, label="relative texture stretch")
ax.set_yscale("log")
ax.set_xlabel("v coordinate"); ax.set_ylabel("relative scale")
ax.set_title("spherical coordinates away from poles and seam")
ax.grid(True, alpha=0.25); ax.legend(fontsize=8)
fig.tight_layout()
param_path = remember(save_matplotlib(fig, TOPIC, "parameterization-distortion-seams.png"), "image")
plt.close(fig)
checks.update({
    "planar_projection_closed_surface_collision_error": planar_collision_error,
    "latlong_pole_to_equator_stretch_ratio": float(latlong_stretch[0].max() / latlong_stretch[len(theta)//2].mean()),
    "cubemap_face_count": int(len(face_counts)),
    "cubemap_all_faces_hit": bool(all(count > 0 for count in face_counts.values())),
})
display_artifact(param_path, width=840)
face_counts

## 3. Interpolated UVs, seams, wrapping modes, and texture transforms

Meshes usually store texture coordinates as per-vertex attributes. Barycentric interpolation then gives a UV for every point in the triangle. This is simple until a triangle crosses a seam. If one vertex has `u = 0.96` and another represents the same neighborhood as `u = 0.04`, direct interpolation walks across almost the whole texture. A seam is represented by duplicating vertices: the two copies have the same 3D position but different UV values.

Wrapping modes extend a finite image to an infinite function. Repeating makes a texture tile; clamping extends edge colors; constant mode creates a background. A texture transform changes scale and placement without changing the mesh UVs.

In [ ]:
t = np.linspace(0, 1, 160)
u_bad = (1 - t) * 0.96 + t * 0.04
u_good_unwrapped = (1 - t) * 0.96 + t * 1.04
u_good_wrapped = u_good_unwrapped % 1.0
v_line = np.full_like(t, 0.48)
out_size = 190
grid = np.linspace(-1.15, 2.15, out_size)
UU, VV = np.meshgrid(grid, grid)
repeat_img = sample_bilinear(texture, UU, VV, mode="repeat")
clamp_img = sample_bilinear(texture, UU, VV, mode="clamp")
constant_img = sample_bilinear(texture, UU, VV, mode="constant", background=(0.94, 0.94, 0.94))
M = np.array([[2.4, 0.28, -0.42], [-0.10, 1.70, 0.22], [0.0, 0.0, 1.0]])
hom = np.stack([UU, VV, np.ones_like(UU)], axis=0).reshape(3, -1)
transformed = (M @ hom).reshape(3, out_size, out_size)
transformed_img = sample_bilinear(texture, transformed[0], transformed[1], mode="repeat")

fig = plt.figure(figsize=(13.0, 8.0))
gs = fig.add_gridspec(2, 4, height_ratios=[1.0, 1.18])
ax = fig.add_subplot(gs[0, 0:2])
ax.imshow(texture, origin="lower", extent=(0, 1, 0, 1))
ax.plot(u_bad, v_line, color="white", lw=4, alpha=0.85, label="shared vertices: long interpolation")
ax.plot(u_bad, v_line, color=COLORS["red"], lw=2)
ax.scatter([0.96, 0.04], [0.48, 0.48], s=90, color=COLORS["red"], edgecolor="white")
ax.set_title("bad seam interpolation crosses the atlas")
ax.set_xlabel("u"); ax.set_ylabel("v"); ax.legend(fontsize=8, loc="lower center")
ax = fig.add_subplot(gs[0, 2:4])
ax.imshow(texture, origin="lower", extent=(0, 1, 0, 1))
ax.plot(u_good_wrapped, v_line, color="white", lw=4, alpha=0.85, label="duplicated seam vertices: local interpolation")
ax.plot(u_good_wrapped, v_line, color=COLORS["green"], lw=2)
ax.scatter([0.96, 0.04], [0.48, 0.48], s=90, color=COLORS["green"], edgecolor="white")
ax.set_title("duplicated seam vertices interpolate locally")
ax.set_xlabel("u"); ax.set_ylabel("v"); ax.legend(fontsize=8, loc="lower center")
for col, (title, img) in enumerate([("repeat", repeat_img), ("clamp", clamp_img), ("constant background", constant_img), ("repeat after UV transform", transformed_img)]):
    ax = fig.add_subplot(gs[1, col])
    ax.imshow(img, origin="lower", extent=(grid.min(), grid.max(), grid.min(), grid.max()))
    ax.add_patch(patches.Rectangle((0, 0), 1, 1, fill=False, edgecolor="white", lw=1.6, ls="--"))
    ax.set_title(title)
    ax.set_xticks([-1, 0, 1, 2]); ax.set_yticks([-1, 0, 1, 2]); ax.set_aspect("equal")
fig.tight_layout()
seam_path = remember(save_matplotlib(fig, TOPIC, "interpolated-uv-seam-wrapping.png"), "image")
plt.close(fig)
checks.update({
    "seam_bad_u_path_length": float(np.abs(u_bad[-1] - u_bad[0])),
    "seam_good_unwrapped_path_length": float(np.abs(u_good_unwrapped[-1] - u_good_unwrapped[0])),
    "seam_duplicate_same_3d_position_different_u": True,
    "wrapping_repeat_periodic_error": float(np.max(np.abs(sample_bilinear(texture, UU + 1.0, VV - 2.0) - repeat_img))),
    "wrapping_clamp_corner_matches_edge": float(np.max(np.abs(clamp_img[0, 0] - sample_bilinear(texture, 0.0, 0.0, mode="clamp")))),
    "texture_transform_matrix_det": float(np.linalg.det(M[:2, :2])),
})
display_artifact(seam_path, width=900)
checks

## 4. Perspective-correct texture interpolation

Rasterizers interpolate attributes across triangles after projection. A texture coordinate that was linear on the 3D triangle is not linear in screen space after the homogeneous divide. The repair is to interpolate `u/w`, `v/w`, and `1/w`, then divide by the interpolated reciprocal depth. This is the same idea that appeared in the graphics pipeline chapter, but here the visual consequence is a warped texture.

In [ ]:
quad_screen = np.array([[-0.82, -0.72], [0.82, -0.72], [0.46, 0.66], [-0.46, 0.66]], dtype=float)
quad_w = np.array([1.0, 1.0, 3.8, 3.8], dtype=float)
quad_uv = np.array([[0.0, 0.0], [3.4, 0.0], [3.4, 4.2], [0.0, 4.2]], dtype=float)
triangles = [(0, 1, 2), (0, 2, 3)]
H, W = 260, 360
ys = np.linspace(-0.88, 0.88, H)
xs = np.linspace(-1.05, 1.05, W)
XX, YY = np.meshgrid(xs, ys)
pts = np.stack([XX, YY], axis=-1)
naive_uv = np.full((H, W, 2), np.nan)
correct_uv = np.full((H, W, 2), np.nan)
mask_total = np.zeros((H, W), dtype=bool)
for tri_idx in triangles:
    tri = quad_screen[list(tri_idx)]
    bary = barycentric_points(pts, tri)
    inside = np.all(bary >= -1e-10, axis=-1)
    uv_tri = quad_uv[list(tri_idx)]
    w_tri = quad_w[list(tri_idx)]
    naive = bary @ uv_tri
    numerator = bary @ (uv_tri / w_tri[:, None])
    denominator = bary @ (1.0 / w_tri)
    corr = numerator / denominator[..., None]
    write = inside & ~mask_total
    naive_uv[write] = naive[write]
    correct_uv[write] = corr[write]
    mask_total |= inside
bg = np.ones((H, W, 3)) * 0.96
naive_img = bg.copy(); correct_img = bg.copy()
naive_img[mask_total] = sample_bilinear(texture, naive_uv[..., 0][mask_total], naive_uv[..., 1][mask_total], mode="repeat")
correct_img[mask_total] = sample_bilinear(texture, correct_uv[..., 0][mask_total], correct_uv[..., 1][mask_total], mode="repeat")
uv_error = np.zeros((H, W))
uv_error[mask_total] = np.linalg.norm(naive_uv[mask_total] - correct_uv[mask_total], axis=1)
fig, axes = plt.subplots(1, 3, figsize=(13.0, 4.2))
for ax, title, img in [(axes[0], "affine screen-space UV", naive_img), (axes[1], "perspective-correct UV", correct_img)]:
    ax.imshow(img, origin="lower", extent=(xs.min(), xs.max(), ys.min(), ys.max()))
    ax.add_patch(patches.Polygon(quad_screen, fill=False, edgecolor="black", lw=1.2))
    ax.set_title(title); ax.set_aspect("equal"); ax.axis("off")
im = axes[2].imshow(uv_error, origin="lower", extent=(xs.min(), xs.max(), ys.min(), ys.max()), cmap="inferno")
axes[2].add_patch(patches.Polygon(quad_screen, fill=False, edgecolor="white", lw=1.2))
axes[2].set_title("UV error from ignoring 1/w"); axes[2].axis("off")
fig.colorbar(im, ax=axes[2], fraction=0.046, pad=0.03)
fig.tight_layout()
perspective_path = remember(save_matplotlib(fig, TOPIC, "affine-vs-perspective-texture.png"), "image")
plt.close(fig)
checks.update({
    "perspective_correct_mask_pixels": int(mask_total.sum()),
    "perspective_correct_max_uv_error_vs_affine": float(np.nanmax(uv_error)),
    "perspective_correct_mean_uv_error_vs_affine": float(np.nanmean(uv_error[mask_total])),
    "perspective_correct_nonzero_error": bool(np.nanmax(uv_error) > 0.25),
})
display_artifact(perspective_path, width=900)
checks

## 5. Pixel footprints, filtering, and mipmaps

A texture-mapped pixel is not a point in texture space. Around an image sample, the mapping from image coordinates `(x, y)` to texture coordinates `(u, v)` has a local Jacobian. The two columns of that Jacobian span an approximate footprint parallelogram. If the footprint is smaller than a texel, reconstruction matters and bilinear interpolation is usually the practical choice. If the footprint covers many texels, the renderer needs an area average.

Mipmaps precompute square averages at powers of two. A lookup estimates a footprint width `D`, chooses level `k = log2(D)`, and interpolates between adjacent levels. This is fast and robust, but a square average is a poor fit for a long grazing-angle footprint, which is why anisotropic filtering samples along the long axis.

In [ ]:
mip_levels = build_mipmap(texture)
base_mean = mip_levels[0].mean(axis=(0, 1))
level_means = np.array([level.mean(axis=(0, 1)) for level in mip_levels])
mean_error = float(np.max(np.abs(level_means - base_mean)))
fig, axes = plt.subplots(2, 4, figsize=(12.5, 6.8))
for ax, level_index in zip(axes.flat[:7], range(7)):
    level = mip_levels[level_index]
    ax.imshow(level, origin="lower")
    ax.set_title(f"level {level_index}: {level.shape[1]} x {level.shape[0]}")
    ax.axis("off")
ax = axes.flat[7]
ax.plot(np.arange(len(mip_levels)), [level.shape[0] for level in mip_levels], "o-", color=COLORS["blue"], label="height")
ax.plot(np.arange(len(mip_levels)), [level.shape[1] for level in mip_levels], "o-", color=COLORS["teal"], label="width")
ax.set_yscale("log", base=2); ax.set_xlabel("mipmap level k"); ax.set_ylabel("texels")
ax.set_title("pyramid dimensions halve each level"); ax.grid(True, alpha=0.25); ax.legend(fontsize=8)
fig.tight_layout()
pyramid_path = remember(save_matplotlib(fig, TOPIC, "mipmap-pyramid-lod.png"), "image")
plt.close(fig)

H, W = 210, 340
sx = np.linspace(-1.0, 1.0, W)
sy = np.linspace(0.0, 1.0, H)
SX, SY = np.meshgrid(sx, sy)
u_map = 0.5 + SX * (1.3 + 5.5 * SY) + 0.07 * np.sin(5.0 * SY)
v_map = 0.2 + 14.5 * SY ** 2
point_img = sample_nearest(texture, u_map, v_map, mode="repeat")
bilinear_img = sample_bilinear(texture, u_map, v_map, mode="repeat")
tex_h, tex_w = texture.shape[:2]
du_dx = np.gradient(u_map, axis=1) * tex_w
dv_dx = np.gradient(v_map, axis=1) * tex_h
du_dy = np.gradient(u_map, axis=0) * tex_w
dv_dy = np.gradient(v_map, axis=0) * tex_h
len_x = np.sqrt(du_dx ** 2 + dv_dx ** 2)
len_y = np.sqrt(du_dy ** 2 + dv_dy ** 2)
D = np.maximum(len_x, len_y)
D_short = np.maximum(np.minimum(len_x, len_y), 1.0)
trilinear_img = sample_mipmap_trilinear(mip_levels, u_map, v_map, D)
long_is_x = len_x >= len_y
long_u = np.where(long_is_x, du_dx / tex_w, du_dy / tex_w)
long_v = np.where(long_is_x, dv_dx / tex_h, dv_dy / tex_h)
aniso_img = np.zeros_like(trilinear_img)
for offset in np.linspace(-0.5, 0.5, 7):
    aniso_img += sample_mipmap_trilinear(mip_levels, u_map + offset * long_u, v_map + offset * long_v, D_short)
aniso_img /= 7.0
far = slice(int(H * 0.62), H)
point_energy_far = image_gradient_energy(point_img[far])
trilinear_energy_far = image_gradient_energy(trilinear_img[far])
aniso_energy_far = image_gradient_energy(aniso_img[far])
fig, axes = plt.subplots(2, 3, figsize=(13.0, 7.7), gridspec_kw={"height_ratios": [1, 1.05]})
for ax, title, img in [(axes[0, 0], "point samples: aliased", point_img), (axes[0, 1], "bilinear reconstruction only", bilinear_img), (axes[0, 2], "trilinear mipmap average", trilinear_img), (axes[1, 0], "anisotropic mipmap samples", aniso_img)]:
    ax.imshow(img, origin="lower"); ax.set_title(title); ax.axis("off")
ax = axes[1, 1]
row = np.arange(H)
ax.plot(row, np.log2(np.maximum(D[:, W // 2], 1.0)), color=COLORS["blue"], label="long-axis LOD")
ax.plot(row, np.log2(np.maximum(D_short[:, W // 2], 1.0)), color=COLORS["teal"], label="short-axis LOD")
ax.set_xlabel("image row"); ax.set_ylabel("log2 footprint width")
ax.set_title("footprint grows toward the horizon"); ax.grid(True, alpha=0.25); ax.legend(fontsize=8)
ax = axes[1, 2]
center = (H // 2, W // 2)
ux = np.array([du_dx[center] / tex_w, dv_dx[center] / tex_h])
uy = np.array([du_dy[center] / tex_w, dv_dy[center] / tex_h])
origin = np.array([0.5, 0.5])
para = np.array([origin - ux/2 - uy/2, origin + ux/2 - uy/2, origin + ux/2 + uy/2, origin - ux/2 + uy/2])
ax.imshow(texture, origin="lower", extent=(0, 1, 0, 1))
ax.add_patch(patches.Polygon(para, closed=True, fill=True, facecolor=(1.0, 0.68, 0.22, 0.35), edgecolor="white", lw=2))
ax.arrow(origin[0], origin[1], ux[0], ux[1], color="white", head_width=0.02, length_includes_head=True)
ax.arrow(origin[0], origin[1], uy[0], uy[1], color="black", head_width=0.02, length_includes_head=True)
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_title("local footprint parallelogram")
ax.set_xlabel("u"); ax.set_ylabel("v")
fig.tight_layout()
filter_path = remember(save_matplotlib(fig, TOPIC, "filtering-mipmap-anisotropy.png"), "image")
plt.close(fig)
checks.update({
    "mipmap_level_count": int(len(mip_levels)),
    "mipmap_last_level_shape": list(mip_levels[-1].shape[:2]),
    "mipmap_mean_preservation_error": mean_error,
    "footprint_lod_min": float(np.log2(np.maximum(D, 1.0)).min()),
    "footprint_lod_max": float(np.log2(np.maximum(D, 1.0)).max()),
    "point_far_gradient_energy": point_energy_far,
    "trilinear_far_gradient_energy": trilinear_energy_far,
    "anisotropic_far_gradient_energy": aniso_energy_far,
    "mipmap_reduces_far_field_gradient_energy": bool(trilinear_energy_far < point_energy_far),
})
display_artifact(pyramid_path, width=820)
display_artifact(filter_path, width=900)
checks

## 6. Normal maps, bump maps, and displacement maps

A normal map changes the normal used in shading. A bump map stores a scalar height field and derives a tangent-frame normal from its derivatives. Both can make lighting react to small features without changing the underlying surface. A displacement map uses the same kind of scalar field differently: it moves geometry, so silhouettes, parallax, and intersections can change.

The static figure compares the height field, normal map, bump-shaded flat patch, and a displaced cross-section. The Plotly surface stores the same displacement as actual `z` coordinates so you can rotate the geometry.

In [ ]:
N = 180
u = np.linspace(0, 1, N); v = np.linspace(0, 1, N)
U, V = np.meshgrid(u, v)
height = (0.035 * np.sin(18 * math.pi * U + 0.8 * np.sin(4 * math.pi * V)) + 0.025 * np.sin(12 * math.pi * V) + 0.095 * np.exp(-((U - 0.68) ** 2 + (V - 0.34) ** 2) / 0.012) - 0.060 * np.exp(-((U - 0.33) ** 2) / 0.0009) * np.exp(-((V - 0.58) ** 2) / 0.08))
dh_dv, dh_du = np.gradient(height, v, u)
normal_tangent = np.dstack([-dh_du, -dh_dv, np.ones_like(height)])
normal_tangent /= np.linalg.norm(normal_tangent, axis=-1, keepdims=True)
normal_map = normal_tangent * 0.5 + 0.5
light = np.array([-0.35, 0.42, 0.84]); light = light / np.linalg.norm(light)
flat_shading = np.clip(np.dot(np.dstack([np.zeros_like(height), np.zeros_like(height), np.ones_like(height)]), light), 0, 1)
bump_shading = np.clip(np.sum(normal_tangent * light, axis=-1), 0, 1)
base_color = sample_bilinear(texture, 2.5 * U, 1.6 * V, mode="repeat")
bump_rgb = base_color * (0.30 + 0.88 * bump_shading[..., None])
flat_rgb = base_color * (0.30 + 0.88 * flat_shading[..., None])
fig, axes = plt.subplots(2, 3, figsize=(13.0, 8.0))
for ax, title, img, kwargs in [
    (axes[0, 0], "bump/displacement height field", height, {"cmap": "terrain"}),
    (axes[0, 1], "tangent-frame normal map", normal_map, {}),
    (axes[0, 2], "same color texture, flat normal", flat_rgb, {}),
    (axes[1, 0], "bump map changes lighting only", bump_rgb, {}),
]:
    ax.imshow(img, origin="lower", **kwargs); ax.set_title(title); ax.axis("off")
row = int(N * 0.58)
axes[1, 1].plot(u, np.zeros_like(u), color=COLORS["gray"], lw=2, label="undisplaced silhouette")
axes[1, 1].plot(u, height[row], color=COLORS["red"], lw=2, label="displaced profile")
axes[1, 1].set_title("displacement changes geometry"); axes[1, 1].set_xlabel("u"); axes[1, 1].set_ylabel("z displacement")
axes[1, 1].grid(True, alpha=0.25); axes[1, 1].legend(fontsize=8)
axes[1, 2].hist(normal_tangent[..., 2].ravel(), bins=28, color=COLORS["teal"], alpha=0.88)
axes[1, 2].set_title("normal z component remains mostly near 1"); axes[1, 2].set_xlabel("n_z"); axes[1, 2].set_ylabel("texels")
fig.tight_layout()
bump_path = remember(save_matplotlib(fig, TOPIC, "bump-displacement-normal-field.png"), "image")
plt.close(fig)
surface_stride = 3
fig3d = go.Figure(data=[go.Surface(x=U[::surface_stride, ::surface_stride], y=V[::surface_stride, ::surface_stride], z=height[::surface_stride, ::surface_stride], surfacecolor=bump_shading[::surface_stride, ::surface_stride], colorscale="Viridis", showscale=True, colorbar=dict(title="bump shade"))])
fig3d.update_layout(title="Displacement map: the height texture is actual geometry here", scene=dict(xaxis_title="u", yaxis_title="v", zaxis_title="height", aspectratio=dict(x=1, y=1, z=0.28)), margin=dict(l=0, r=0, t=45, b=0), height=620)
displacement_html = remember(save_plotly_html(fig3d, TOPIC, "bump-displacement-surface.html"), "html")
checks.update({
    "normal_map_unit_length_max_error": float(np.max(np.abs(np.linalg.norm(normal_tangent, axis=-1) - 1.0))),
    "bump_changes_lighting_mean_abs_delta": float(np.mean(np.abs(bump_rgb - flat_rgb))),
    "displacement_height_range": float(height.max() - height.min()),
    "displacement_profile_peak_to_peak": float(height[row].max() - height[row].min()),
})
display_artifact(bump_path, width=900)
display_artifact(displacement_html, width="100%", height=560)
checks

## 7. Shadow maps are depth textures from the light

A shadow map is a texture whose value is depth from the light. During the main render, the fragment is projected into the light's texture coordinates and its light-space depth is compared with the stored closest depth. Bias is required because the same visible surface may compare slightly behind itself after quantization and interpolation. Near a depth discontinuity, interpolate the binary closer test rather than the depths; this is percentage-closer filtering.

In [ ]:
n = 240
s = np.linspace(0, 1, n)
shadow_map = np.full(n, 2.0)
blocker = (s > 0.43) & (s < 0.58)
shadow_map[blocker] = 1.08 + 0.04 * np.cos((s[blocker] - 0.50) * 16 * math.pi)
receiver_depth = np.full(n, 2.0) + 0.006 * (0.55 + 0.45 * np.sin(90 * s))
bias = 0.012
lit_no_bias = receiver_depth - shadow_map <= 0.0
lit_with_bias = receiver_depth - shadow_map <= bias

def shadow_test_at(offset):
    idx = np.clip(np.round((s + offset) * (n - 1)).astype(int), 0, n - 1)
    return (receiver_depth - shadow_map[idx]) <= bias
pcf = np.zeros(n, dtype=float)
for offset in np.linspace(-0.018, 0.018, 7):
    pcf += shadow_test_at(offset).astype(float)
pcf /= 7.0
fig, axes = plt.subplots(3, 1, figsize=(12.0, 8.0), sharex=True)
axes[0].plot(s, shadow_map, color=COLORS["blue"], lw=2, label="stored closest depth")
axes[0].plot(s, receiver_depth, color=COLORS["red"], lw=1.6, label="fragment depth")
axes[0].fill_between(s, 0.95, 2.15, where=blocker, color=COLORS["gray"], alpha=0.15, label="occluder in light map")
axes[0].set_title("shadow lookup is a depth comparison"); axes[0].set_ylabel("light-space depth")
axes[0].legend(fontsize=8, loc="lower left"); axes[0].grid(True, alpha=0.25)
for y0, arr, label, color in [(0.0, lit_no_bias, "no bias", COLORS["red"]), (1.0, lit_with_bias, "bias", COLORS["teal"]), (2.0, pcf, "PCF average", COLORS["gold"])]:
    strip = np.tile(arr.astype(float) if arr.dtype == bool else arr, (18, 1))
    axes[1].imshow(strip, extent=(0, 1, y0, y0 + 0.65), origin="lower", aspect="auto", cmap="gray", vmin=0, vmax=1)
    axes[1].text(-0.02, y0 + 0.32, label, ha="right", va="center", color=color)
axes[1].set_ylim(-0.1, 2.8); axes[1].set_yticks([]); axes[1].set_title("white means lit; PCF averages visibility tests")
axes[2].plot(s, lit_no_bias.astype(float), color=COLORS["red"], lw=1.4, label="no bias")
axes[2].plot(s, lit_with_bias.astype(float), color=COLORS["teal"], lw=1.6, label="with bias")
axes[2].plot(s, pcf, color=COLORS["gold"], lw=2.0, label="PCF")
axes[2].set_xlabel("shadow-map coordinate s"); axes[2].set_ylabel("visibility"); axes[2].set_ylim(-0.05, 1.05)
axes[2].grid(True, alpha=0.25); axes[2].legend(fontsize=8)
fig.tight_layout()
shadow_path = remember(save_matplotlib(fig, TOPIC, "shadow-map-bias-pcf.png"), "image")
plt.close(fig)
lit_region = ~blocker
checks.update({
    "shadow_no_bias_lit_fraction_on_lit_region": float(lit_no_bias[lit_region].mean()),
    "shadow_bias_lit_fraction_on_lit_region": float(lit_with_bias[lit_region].mean()),
    "shadow_bias_shadow_fraction_under_blocker": float((~lit_with_bias[blocker]).mean()),
    "pcf_min": float(pcf.min()),
    "pcf_max": float(pcf.max()),
    "pcf_has_fractional_boundary_values": bool(np.any((pcf > 0.0) & (pcf < 1.0))),
})
display_artifact(shadow_path, width=850)
checks

## 8. Environment maps: texture lookup by direction

An environment map stores a function on directions. If a ray misses the scene, or if a shiny surface asks what it reflects, the lookup coordinate comes from a unit vector rather than a surface position. Longitude-latitude maps are easy to write down, but they compress samples near the poles; cubemaps reduce that distortion by using six charts.

The synthetic map below has a bright sun spot, a warm horizon band, and a darker ground band. The sphere image samples the map with reflection vectors from a mirror sphere.

In [ ]:
env_h, env_w = 180, 360
lon = np.linspace(-math.pi, math.pi, env_w, endpoint=False)
lat = np.linspace(-math.pi / 2, math.pi / 2, env_h)
LON, LAT = np.meshgrid(lon, lat)
sky = 0.5 + 0.5 * np.sin(LAT)
horizon = np.exp(-(LAT / 0.18) ** 2)
sun = np.exp(-((LON - 0.78) ** 2 / 0.035 + (LAT - 0.42) ** 2 / 0.020))
ground = np.clip(-np.sin(LAT), 0, 1)
env = np.dstack([0.10 + 0.22 * sky + 0.78 * sun + 0.28 * horizon, 0.18 + 0.36 * sky + 0.58 * sun + 0.20 * ground, 0.30 + 0.58 * sky + 0.18 * sun + 0.08 * ground])
env = np.clip(env, 0, 1)

def direction_to_lonlat_uv(direction):
    d = np.asarray(direction, dtype=float)
    d = d / np.linalg.norm(d, axis=-1, keepdims=True)
    lon = np.arctan2(d[..., 1], d[..., 0])
    lat = np.arcsin(np.clip(d[..., 2], -1.0, 1.0))
    u = (lon + math.pi) / (2 * math.pi)
    v = (lat + math.pi / 2) / math.pi
    return u, v
S = 260
sx = np.linspace(-1, 1, S); sy = np.linspace(-1, 1, S)
X, Y = np.meshgrid(sx, sy)
r2 = X ** 2 + Y ** 2
sphere_mask = r2 <= 1.0
Z = np.sqrt(np.clip(1.0 - r2, 0.0, 1.0))
normal = np.dstack([X, Y, Z])
view_dir = np.array([0.0, 0.0, 1.0])
reflection = 2.0 * np.sum(normal * view_dir, axis=-1, keepdims=True) * normal - view_dir
reflection_norm = np.linalg.norm(reflection[sphere_mask], axis=-1)
ru, rv = direction_to_lonlat_uv(reflection)
sphere_img = np.ones((S, S, 3)) * 0.96
sphere_img[sphere_mask] = sample_bilinear(env, ru[sphere_mask], rv[sphere_mask], mode="repeat")
sphere_img[sphere_mask] *= (0.50 + 0.50 * np.clip(normal[..., 2][sphere_mask], 0, 1))[..., None]
uv_debug = np.dstack([ru, rv, 0.25 + 0.75 * sphere_mask.astype(float)])
uv_debug[~sphere_mask] = 1.0
fig, axes = plt.subplots(1, 3, figsize=(13.0, 4.0))
axes[0].imshow(env, origin="lower", extent=(-180, 180, -90, 90)); axes[0].set_title("synthetic longitude-latitude environment")
axes[0].set_xlabel("longitude"); axes[0].set_ylabel("latitude")
axes[1].imshow(uv_debug, origin="lower"); axes[1].set_title("reflection vector converted to (u, v)"); axes[1].axis("off")
axes[2].imshow(sphere_img, origin="lower"); axes[2].set_title("mirror sphere samples the environment"); axes[2].axis("off")
fig.tight_layout()
env_path = remember(save_matplotlib(fig, TOPIC, "environment-map-reflection-lookup.png"), "image")
plt.close(fig)
checks.update({
    "environment_reflection_unit_max_error": float(np.max(np.abs(reflection_norm - 1.0))),
    "environment_uv_min": float(min(ru[sphere_mask].min(), rv[sphere_mask].min())),
    "environment_uv_max": float(max(ru[sphere_mask].max(), rv[sphere_mask].max())),
    "environment_map_bright_sun_contrast": float(env.max() - env.mean()),
})
display_artifact(env_path, width=900)
checks

## 9. Procedural 3D textures

A procedural texture replaces a stored 2D image with a function of position. That is useful when the material should behave like a solid medium, because the texture is already defined for every point in space. A stripe texture is a simple periodic function. Smooth solid noise is more useful for natural variation: random gradients live on a lattice, Hermite weights interpolate them smoothly, and turbulence adds scaled absolute-value noise across octaves.

In [ ]:
rng = np.random.default_rng(1107)
perm = rng.permutation(256)
raw_grads = rng.normal(size=(256, 3))
grads = raw_grads / np.linalg.norm(raw_grads, axis=1, keepdims=True)

def fade(t):
    return t * t * (3.0 - 2.0 * t)

def grad_hash(ix, iy, iz):
    return perm[(ix + perm[(iy + perm[iz % 256]) % 256]) % 256]

def gradient_noise_3d(points):
    pnts = np.asarray(points, dtype=float)
    cell = np.floor(pnts).astype(int)
    f = pnts - cell
    result = np.zeros(pnts.shape[:-1], dtype=float)
    wx = fade(f[..., 0]); wy = fade(f[..., 1]); wz = fade(f[..., 2])
    for dx in [0, 1]:
        for dy in [0, 1]:
            for dz in [0, 1]:
                idx = grad_hash(cell[..., 0] + dx, cell[..., 1] + dy, cell[..., 2] + dz)
                disp = f - np.array([dx, dy, dz])
                dot = np.sum(grads[idx] * disp, axis=-1)
                weight = (wx if dx else 1 - wx) * (wy if dy else 1 - wy) * (wz if dz else 1 - wz)
                result += weight * dot
    return result

def turbulence(points, octaves=6):
    total = np.zeros(points.shape[:-1], dtype=float)
    amplitudes = []
    for i in range(octaves):
        amp = 1.0 / (2 ** i)
        amplitudes.append(amp)
        total += amp * np.abs(gradient_noise_3d(points * (2 ** i)))
    return total, np.array(amplitudes)
N = 220
x = np.linspace(0, 6.0, N); y = np.linspace(0, 6.0, N)
X, Y = np.meshgrid(x, y)
P = np.dstack([X, Y, np.full_like(X, 0.37)])
stripe_hard = (np.sin(math.pi * X / 0.42) > 0).astype(float)
stripe_smooth = 0.5 + 0.5 * np.sin(math.pi * X / 0.42)
noise = gradient_noise_3d(P * 0.85)
turb, amplitudes = turbulence(P * 0.72, octaves=6)
turb_stripe = 0.5 + 0.5 * np.sin(2.2 * X + 5.2 * turb)
color0 = np.array([0.14, 0.22, 0.38]); color1 = np.array([0.86, 0.62, 0.25])
hard_rgb = color0 * (1 - stripe_hard[..., None]) + color1 * stripe_hard[..., None]
smooth_rgb = color0 * (1 - stripe_smooth[..., None]) + color1 * stripe_smooth[..., None]
noise_rgb = plt.get_cmap("cividis")(normalize01(noise))[..., :3]
turb_rgb = plt.get_cmap("magma")(normalize01(turb))[..., :3]
turb_stripe_rgb = color0 * (1 - turb_stripe[..., None]) + color1 * turb_stripe[..., None]
fig, axes = plt.subplots(2, 3, figsize=(12.5, 8.0))
for ax, title, img in [(axes[0, 0], "hard sine stripes", hard_rgb), (axes[0, 1], "smooth sine stripes", smooth_rgb), (axes[0, 2], "gradient solid noise slice", noise_rgb), (axes[1, 0], "absolute-value turbulence", turb_rgb), (axes[1, 1], "turbulence-distorted stripes", turb_stripe_rgb)]:
    ax.imshow(img, origin="lower", extent=(0, 6, 0, 6)); ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])
ax = axes[1, 2]
octave_variance = []
for i, amp in enumerate(amplitudes):
    component = amp * np.abs(gradient_noise_3d(P * 0.72 * (2 ** i)))
    octave_variance.append(float(np.var(component)))
ax.plot(np.arange(len(amplitudes)), amplitudes, "o-", color=COLORS["blue"], label="amplitude")
ax.plot(np.arange(len(octave_variance)), octave_variance, "o-", color=COLORS["red"], label="component variance")
ax.set_title("turbulence adds smaller octaves"); ax.set_xlabel("octave"); ax.grid(True, alpha=0.25); ax.legend(fontsize=8)
fig.tight_layout()
procedural_path = remember(save_matplotlib(fig, TOPIC, "procedural-solid-texture-turbulence.png"), "image")
plt.close(fig)
checks.update({
    "procedural_noise_min": float(noise.min()),
    "procedural_noise_max": float(noise.max()),
    "procedural_noise_finite": bool(np.all(np.isfinite(noise))),
    "turbulence_amplitudes_strictly_decrease": bool(np.all(np.diff(amplitudes) < 0)),
    "turbulence_component_variance_decreases_after_first": bool(octave_variance[-1] < octave_variance[0]),
    "turbulent_stripe_std": float(np.std(turb_stripe)),
})
display_artifact(procedural_path, width=900)
checks

## Applied lab

Use the notebook as a texture-mapping debugger.

1. Change the seam example so the right endpoint is `u = 0.20` instead of `u = 0.04`. Re-run the seam cell and decide whether duplicating vertices is still the correct fix or whether the atlas edge no longer represents a seam crossing.
2. In the mipmap section, replace `D = max(len_x, len_y)` with `D = sqrt(len_x * len_y)`. Compare the far-field gradient energy and the visible blur. This tests the tradeoff between area-based and long-axis level selection.
3. Increase the shadow-map `bias` until blocker shadows start to detach. A good answer reports both a visual observation and a changed check, such as a smaller `shadow_bias_shadow_fraction_under_blocker`.
4. In the procedural section, double the stripe frequency and halve the turbulence scale. The texture should become more regular; the `turbulent_stripe_std` and the visible stripe bending should change together.

## Display the chapter artifacts

The chapter artifact set is concept-named and lives under `artifacts/chapter-11`. The PNG files are durable snapshots for audits; the Plotly HTML file is included where 3D rotation teaches something the static images cannot.

In [ ]:
artifact_sequence = [*image_paths, *html_paths]
assert_artifacts(artifact_sequence)
for path in image_paths:
    display_artifact(path, width=760)
for path in html_paths:
    display_artifact(path, width="100%", height=560)
[rel_book(path) for path in artifact_sequence]

## Sanity checks

These checks are the notebook's compact texture-mapping contract:

- texture lookup coordinates stay in bounds when clamped and repeat mode is periodic,
- analytic parameterizations expose expected distortion and cubemap face coverage,
- seam duplication changes UV interpolation from a long atlas path to a local path,
- perspective-correct interpolation differs from naive screen-linear interpolation on a projected quad,
- mipmaps preserve the mean and reduce far-field high-frequency energy,
- bump-map normals remain unit length and displacement produces actual height variation,
- shadow-map bias repairs self-shadowing while PCF produces fractional boundary visibility,
- environment-map reflection vectors remain unit length and map to valid UVs,
- procedural noise and turbulence are finite, nonconstant fields.

The JSON and CSV artifacts are part of the lesson: they record what the visuals are supposed to demonstrate so later edits can be audited without guessing the intent.

In [ ]:
assert 0.0 <= checks["texture_lookup_uv_min"] <= checks["texture_lookup_uv_max"] <= 1.0
assert checks["repeat_mode_periodic_error"] < 1e-12
assert checks["diagnostic_texture_channel_std_min"] > 0.10
assert checks["planar_projection_closed_surface_collision_error"] < 1e-12
assert checks["latlong_pole_to_equator_stretch_ratio"] > 100.0
assert checks["cubemap_all_faces_hit"] is True
assert checks["seam_bad_u_path_length"] > 0.8
assert checks["seam_good_unwrapped_path_length"] < 0.09
assert checks["wrapping_repeat_periodic_error"] < 1e-12
assert abs(checks["texture_transform_matrix_det"]) > 1.0
assert checks["perspective_correct_mask_pixels"] > 10000
assert checks["perspective_correct_nonzero_error"] is True
assert checks["mipmap_level_count"] >= 8
assert checks["mipmap_last_level_shape"] == [1, 1]
assert checks["mipmap_mean_preservation_error"] < 1e-12
assert checks["footprint_lod_max"] > checks["footprint_lod_min"]
assert checks["mipmap_reduces_far_field_gradient_energy"] is True
assert checks["normal_map_unit_length_max_error"] < 1e-12
assert checks["bump_changes_lighting_mean_abs_delta"] > 0.01
assert checks["displacement_height_range"] > 0.12
assert checks["shadow_bias_lit_fraction_on_lit_region"] > checks["shadow_no_bias_lit_fraction_on_lit_region"]
assert checks["shadow_bias_shadow_fraction_under_blocker"] > 0.95
assert 0.0 <= checks["pcf_min"] <= checks["pcf_max"] <= 1.0
assert checks["pcf_has_fractional_boundary_values"] is True
assert checks["environment_reflection_unit_max_error"] < 1e-12
assert 0.0 <= checks["environment_uv_min"] <= checks["environment_uv_max"] <= 1.0
assert checks["procedural_noise_finite"] is True
assert checks["turbulence_amplitudes_strictly_decrease"] is True
assert checks["turbulence_component_variance_decreases_after_first"] is True
assert checks["turbulent_stripe_std"] > 0.20

image_records = [assert_nonblank_image(path) for path in image_paths]
artifact_records = assert_artifacts([*image_paths, *html_paths])
check_rows = [{"check": key, "value": str(value), "concept": "texture mapping invariant"} for key, value in sorted(checks.items())]
ledger_path = remember(save_table_csv(check_rows, TOPIC, "texture-mapping-check-ledger.csv"), "table")
storyboard = {
    "chapter": CHAPTER,
    "title": TITLE,
    "source_span": SOURCE_SPAN,
    "visuals": [rel_book(path) for path in image_paths + html_paths],
    "library_routing": {
        "numpy": "texture arrays, UV maps, barycentric interpolation, Jacobians, mipmaps, shadow/depth comparisons, reflection vectors, procedural noise",
        "matplotlib": "durable 2D diagrams, atlas diagnostics, sampler comparisons, shadow-map and procedural snapshots",
        "plotly": "interactive 3D displacement surface so changed geometry can be rotated and inspected",
        "pandas": "check ledger table saved as CSV",
    },
    "checks": checks,
}
storyboard_path = remember(save_json(storyboard, TOPIC, "visual-storyboard.json"), "check")
numeric_path = remember(save_json({"chapter": CHAPTER, "title": TITLE, "source_span": SOURCE_SPAN, "checks": checks}, TOPIC, "numeric-checks.json"), "check")
artifact_sanity = {
    "chapter": CHAPTER,
    "artifact_count_before_sanity_file": len(image_paths) + len(html_paths) + len(table_paths) + len(check_paths),
    "artifacts": artifact_records,
    "nonblank_images": image_records,
    "checks_json": rel_book(numeric_path),
}
artifact_sanity_path = remember(save_json(artifact_sanity, TOPIC, "artifact-sanity.json"), "check")
final_report = {
    "chapter": CHAPTER,
    "title": TITLE,
    "printed_pages": PRINTED_PAGES,
    "pdf_pages": PDF_PAGES,
    "artifacts": [rel_book(path) for path in image_paths + html_paths + table_paths + check_paths + [ARTIFACT_ROOT / "checks" / "final-sanity.json"]],
    "nonblank_images": len(image_records),
    "checks": checks,
    "notebook_executed": True,
}
final_path = remember(save_json(final_report, TOPIC, "final-sanity.json"), "check")
assert_artifacts([*table_paths, *check_paths])
display_artifact(numeric_path)
display_artifact(storyboard_path)
display_artifact(final_path)
final_report

## Takeaways

- Texture mapping is a lookup pipeline: surface point, texture coordinate function, filtered texture value, then shading or geometry use.
- Parameterization is geometry. Planar projection, latitude-longitude maps, cylindrical maps, cubemaps, and mesh atlases trade continuity, distortion, and injectivity differently.
- Seam handling is explicit in meshes: duplicate vertices when the same 3D position needs different UV coordinates on opposite sides of a seam.
- Perspective-correct interpolation is not optional for textured projected triangles; screen-linear UVs are visibly wrong when depth varies.
- Texture filtering is Chapter 10 resampling in disguise, with a moving footprint. Mipmaps are fast square averages; anisotropic filtering exists because many footprints are long and thin.
- Normal and bump maps change shading normals. Displacement maps change geometry.
- Shadow maps and environment maps are texture lookups with non-color meanings: depth from a light and radiance by direction.
- Procedural solid textures avoid 2D parameterization for materials that are naturally functions of 3D position.